In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

## Sección 1 — El ecosistema BERT

### ¿Qué es BERT y qué problema resuelve?

Antes de BERT, los modelos de NLP representaban las palabras con **embeddings estáticos** (Word2Vec, GloVe): cada palabra tenía un único vector independientemente del contexto. La palabra "banco" tenía la misma representación en "banco financiero" y en "banco de madera".

BERT (*Bidirectional Encoder Representations from Transformers*, Devlin et al. 2018) resuelve esto con **representaciones contextuales**: el vector de cada token depende de todos los tokens que lo rodean, procesados simultáneamente en ambas direcciones gracias a la arquitectura Transformer con atención bidireccional.

**Preentrenamiento:** BERT se entrena con dos tareas auto-supervisadas sobre grandes corpus de texto:

- **MLM (Masked Language Modeling):** se ocultan aleatoriamente el 15% de los tokens y el modelo aprende a predecirlos usando el contexto izquierdo y derecho. Esto fuerza al modelo a construir representaciones ricas de cada palabra.
- **NSP (Next Sentence Prediction):** el modelo aprende si dos oraciones son consecutivas o aleatorias. Captura relaciones entre frases.

El resultado es un modelo que ya "entiende" el lenguaje y puede adaptarse a tareas específicas con muy poco datos adicionales mediante **fine-tuning**.

### Variantes principales de BERT

La familia BERT se ha expandido con modelos especializados. La elección depende del dominio y los recursos disponibles:

| Modelo | Preentrenado en | Cuándo usarlo |
|---|---|---|
| **BERT base** | Wikipedia + BooksCorpus (3.3B tokens) | Punto de partida, propósito general |
| **RoBERTa** | Más datos, más tiempo, sin NSP | Cuando BERT base no basta |
| **DistilBERT** | Destilado de BERT (40% menos parámetros) | Velocidad > precisión, recursos limitados |
| **SciBERT** | 1.14M papers de Semantic Scholar | Textos académicos y científicos |
| **BioBERT** | Literatura médica PubMed + PMC | Dominio biomédico específico |

### ¿Por qué SciBERT para este proyecto?

Los abstracts de arXiv están repletos de terminología científica que BERT base trata como tokens raros o fragmenta en subpalabras:

- Términos como `ablation`, `backpropagation`, `tokenization`, `convolutional` son **tokens nativos** en SciBERT porque su vocabulario fue construido desde cero sobre 1.14M papers de Semantic Scholar (de las áreas de Computer Science y Biomedicine).
- En BERT base, esos mismos términos se fragmentan: `back` + `##prop` + `##agation`. El modelo pierde información sobre la estructura del término.

**Consecuencia práctica:** SciBERT produce representaciones más precisas de los abstracts académicos, lo que se traduce en mejor rendimiento en clasificación de papers comparado con BERT base, sin necesidad de más datos ni más epochs de fine-tuning.

## Sección 2 — Qué es fine-tuning y cómo funciona

### Transfer learning: partir del conocimiento existente

**Transfer learning** consiste en tomar un modelo ya entrenado para una tarea general y adaptarlo a una tarea específica. La analogía médica es útil: un médico general que se especializa en cardiología no vuelve a aprender anatomía desde cero. Parte de un conocimiento sólido del cuerpo humano y aprende las particularidades del corazón.

SciBERT ya sabe qué es un paper científico, reconoce términos técnicos y entiende la estructura de los abstracts. El fine-tuning le enseña a distinguir entre categorías de papers (`cs.AI` vs `cs.CV` vs `cs.SE`), una tarea mucho más acotada que aprender el lenguaje científico desde cero.

Con 1500 artículos de entrenamiento (150 por clase) y solo 3 epochs, se obtienen resultados competitivos precisamente porque no partimos de cero.

### Qué capas se modifican y cuáles se preservan

La arquitectura durante el fine-tuning tiene dos partes con roles distintos:

**Capas del encoder BERT (12 capas Transformer):**  
Contienen el conocimiento lingüístico preentrenado. Se actualizan con un **learning rate muy bajo** (2e-5) para refinar sutilmente las representaciones sin borrar lo aprendido. Este riesgo se llama *catastrophic forgetting*: un learning rate alto haría que el modelo "olvide" el lenguaje científico aprendido en el preentrenamiento.

**Capa clasificadora (una capa linear sobre el token `[CLS]`):**  
Es nueva, inicializada aleatoriamente. Aprende a mapear la representación del `[CLS]` (resumen del documento) a los 10 logits de clase. Esta capa necesita aprender desde cero, pero lo hace rápido porque recibe representaciones ya ricas de las capas BERT.

El token `[CLS]` se usa como representación del documento completo porque BERT lo entrena explícitamente con NSP para condensar el significado de toda la secuencia.

In [ ]:
# Cargamos el modelo entrenado con abstract_api para inspeccionar su arquitectura.
# El modelo está guardado en formato safetensors con la configuración correcta
# de id2label y label2id incorporada desde el entrenamiento.
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "models/scibert_api/best_model"
)

# Contar parámetros: totales vs solo el clasificador
total_params      = sum(p.numel() for p in model.parameters())
classifier_params = sum(p.numel() for p in model.classifier.parameters())
bert_params       = total_params - classifier_params

print(f"Arquitectura: {model.config.model_type} para clasificación de secuencias")
print(f"Número de clases: {model.config.num_labels}")
print(f"Clases: {list(model.config.id2label.values())}")
print()
print(f"{'Componente':<30} {'Parámetros':>15} {'%':>8}")
print("-" * 56)
print(f"{'Encoder SciBERT (12 capas)':<30} {bert_params:>15,} {100*bert_params/total_params:>7.2f}%")
print(f"{'Clasificador (linear)':<30} {classifier_params:>15,} {100*classifier_params/total_params:>7.2f}%")
print("-" * 56)
print(f"{'TOTAL':<30} {total_params:>15,} {'100.00%':>8}")

## Sección 3 — Configuración del entrenamiento

### Hiperparámetros y su justificación

Cada hiperparámetro del loop de entrenamiento tiene una razón de ser:

| Hiperparámetro | Valor | Justificación |
|---|---|---|
| `learning_rate` | 2e-5 | Rango estándar para fine-tuning de BERT (1e-5–5e-5). Valores más altos causan *catastrophic forgetting*; más bajos, convergencia lenta. |
| `warmup_steps` | 10% del total | El scheduler sube el LR gradualmente durante el warm-up y lo decae linealmente después. Evita inestabilidad en los primeros pasos cuando los gradientes son ruidosos. |
| `weight_decay` | 0.01 | Regularización L2 aplicada en AdamW. Penaliza pesos grandes y reduce overfitting sin afectar los sesgos. |
| `accumulation_steps` | 2 | Acumula gradientes de 2 batches antes de actualizar. El batch efectivo es el doble del físico, mejorando la estabilidad del gradiente sin requerir más GPU RAM. |
| `grad_clip_norm` | 1.0 | Recorta el gradiente si su norma supera 1.0. Previene *exploding gradients*, especialmente en los primeros pasos del entrenamiento. |
| `epochs` | 3 | Estándar para fine-tuning de BERT (2–4 epochs). Más epochs suelen generar overfitting porque el dataset de entrenamiento es pequeño (1500 artículos). |

### Configuración de hardware

El entrenamiento detecta automáticamente la GPU disponible y ajusta el batch size según la memoria:

| Memoria GPU | Batch size físico | Batch size efectivo (× accumulation) |
|---|---|---|
| ≥ 8 GB | 32 | 64 |
| ≥ 6 GB | 16 | 32 |
| CPU / < 6 GB | 8 | 16 |

SciBERT con secuencias de 512 tokens requiere ~6 GB de VRAM con batch size 16. El gradient accumulation permite simular batches más grandes sin aumentar el pico de memoria, lo que es especialmente útil en GPUs con menos de 8 GB.

Los tres modelos (`api`, `pymupdf`, `docling`) se entrenaron con la misma configuración de hardware para garantizar comparabilidad de tiempos y resultados.

## Sección 4 — Resultados del entrenamiento

In [ ]:
# Cargamos los historiales de entrenamiento de los 3 modelos.
# Cada archivo contiene una lista de dicts con métricas por epoch:
# epoch, train_loss, val_loss, val_accuracy, val_f1_macro, val_f1_weighted
sources = ["api", "pymupdf", "docling"]
histories = {}

for source in sources:
    path = Path(f"reports/training_history_{source}.json")
    with open(path) as f:
        histories[source] = json.load(f)

# Mostramos los primeros registros para verificar la estructura
print("Estructura de training_history_api.json:")
print(f"  Campos: {list(histories['api'][0].keys())}")
print(f"  Epochs registrados: {[h['epoch'] for h in histories['api']]}")

In [ ]:
# Tabla comparativa con las métricas del epoch final de cada modelo.
# El epoch final es siempre el epoch 3 (o el último disponible en test mode).
# El mejor modelo según val_f1_macro se guarda en best_model/ durante el entrenamiento.
print(f"{'Fuente':<10} {'Epochs':>7} {'Train loss':>12} {'Val loss':>10} {'Val F1 macro':>13}")
print("-" * 56)

for source in sources:
    history = histories[source]
    last    = history[-1]   # métricas del último epoch
    best_f1 = max(h["val_f1_macro"] for h in history)
    print(
        f"{source:<10}"
        f"{len(history):>7}"
        f"{last['train_loss']:>12.4f}"
        f"{last['val_loss']:>10.4f}"
        f"{best_f1:>13.4f}  ← mejor epoch"
    )

In [ ]:
# Curvas de entrenamiento: train_loss y val_f1_macro por epoch para los 3 modelos.
# Permiten diagnosticar si el entrenamiento convergió, si hubo overfitting
# (val_loss sube mientras train_loss baja) o si hacen falta más epochs.

styles = {
    "api":     {"color": "steelblue",  "marker": "o", "label": "abstract_api"},
    "pymupdf": {"color": "darkorange", "marker": "s", "label": "abstract_pymupdf"},
    "docling": {"color": "seagreen",   "marker": "^", "label": "abstract_docling"},
}

fig, (ax_loss, ax_f1) = plt.subplots(1, 2, figsize=(12, 5))

for source, style in styles.items():
    history = histories[source]
    epochs      = [h["epoch"]        for h in history]
    train_loss  = [h["train_loss"]   for h in history]
    val_f1      = [h["val_f1_macro"] for h in history]

    ax_loss.plot(epochs, train_loss, marker=style["marker"],
                 color=style["color"], label=style["label"], linewidth=1.8)
    ax_f1.plot(epochs, val_f1, marker=style["marker"],
               color=style["color"], label=style["label"], linewidth=1.8)

ax_loss.set_title("Train loss por epoch")
ax_loss.set_xlabel("Epoch")
ax_loss.set_ylabel("Loss")
ax_loss.set_xticks([h["epoch"] for h in histories["api"]])
ax_loss.legend()

ax_f1.set_title("Val F1 macro por epoch")
ax_f1.set_xlabel("Epoch")
ax_f1.set_ylabel("F1 macro")
ax_f1.set_xticks([h["epoch"] for h in histories["api"]])
ax_f1.legend()

fig.suptitle("Curvas de entrenamiento — SciBERT fine-tuning (3 fuentes)", fontsize=13)
plt.tight_layout()
plt.show()

### Interpretación de las curvas

**Train loss decreciente:** confirma que el modelo está aprendiendo en cada epoch. Una train loss que no baja indicaría un learning rate demasiado bajo o un bug en el loop de entrenamiento.

**Val F1 creciente y estable:** indica que el modelo generaliza. Si val_f1 subiera en epoch 1 y 2 pero cayera en epoch 3 mientras train_loss sigue bajando, estaríamos ante **overfitting**: el modelo memoriza el train set en lugar de aprender patrones generalizables. Con solo 1500 artículos de entrenamiento este riesgo es real, por eso se usan regularización (weight_decay) y early stopping implícito (guardamos el mejor checkpoint, no el último).

**¿Por qué `abstract_api` debería rendir mejor?**  
El abstract de la API viene directamente de los metadatos oficiales del paper: es el texto exacto que el autor publicó, sin artefactos de extracción. PyMuPDF y Docling reconstruyen el abstract desde el PDF, donde pueden aparecer guiones de separación de sílabas, caracteres especiales o texto de otras secciones. Cualquier ruido en el texto de entrada es ruido que el modelo debe ignorar, lo que dificulta el aprendizaje.

## Sección 5 — Punto clave

**Fine-tuning aprovecha el conocimiento previo de SciBERT**  
No entrenamos un clasificador desde cero. Partimos de un modelo que ya comprende el lenguaje científico (terminología, estructura de abstracts, relaciones semánticas entre conceptos) y le enseñamos únicamente la tarea de discriminar entre 10 categorías. Esto hace posible obtener resultados sólidos con 1500 artículos de entrenamiento.

**Solo 3 epochs son suficientes**  
El dominio (papers de Computer Science) ya está cubierto en el preentrenamiento de SciBERT. El fine-tuning ajusta sutilmente las representaciones, no las construye desde cero. Más de 3–4 epochs suelen empeorar el rendimiento en datasets pequeños porque el modelo empieza a memorizar.

**La calidad del texto de entrada impacta el rendimiento**  
Hemos entrenado tres modelos idénticos en arquitectura, hiperparámetros y datos, con una única diferencia: la fuente del abstract. Cualquier diferencia en val_f1_macro es atribuible a la calidad del texto extraído. Esto convierte el experimento en una evaluación controlada de los extractores de PDF.

**En la siguiente etapa veremos qué tan grande es ese impacto**  
El notebook `06_evaluation.ipynb` cargará los tres modelos entrenados y los evaluará sobre el mismo test set, produciendo métricas por clase y matrices de confusión que revelarán no solo cuál fuente es mejor, sino en qué categorías las diferencias son más pronunciadas.